# PRISM - CONCH + LungHist700
# Pathology Reliability In Scarce-label Medicine

In [1]:
import torch
import numpy as np
import pandas as pd
import os
from google.colab import userdata, drive
from torchvision import transforms

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/conch/lunghist700'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


In [2]:
!pip install transformers timm einops huggingface_hub scikit-learn scipy tqdm -q
!pip install git+https://github.com/Mahmoodlab/CONCH.git -q

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Login OK")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
Login OK


In [3]:
from conch.open_clip_custom import create_model_from_pretrained
model, transform = create_model_from_pretrained(
    'conch_ViT-B-16',
    checkpoint_path='hf_hub:MahmoodLab/CONCH',
    hf_auth_token=userdata.get('HF_TOKEN')
)
model = model.cuda().eval()
print("CONCH loaded!")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

CONCH loaded!


In [4]:
# CONCH provides its own transform above

In [5]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

# LungHist700: 7 classes inside images/ folder
# 70% train / 15% val / 15% test
DATASET_DIR = '/content/drive/MyDrive/PRISM/datasets/lunghist700/images'
full_dataset = ImageFolder(root=DATASET_DIR, transform=transform)

train_size = int(0.70 * len(full_dataset))
val_size   = int(0.15 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Classes: {full_dataset.classes}")
print(f"Total: {len(full_dataset):,}")
print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

Classes: ['aca_bd', 'aca_md', 'aca_pd', 'nor', 'scc_bd', 'scc_md', 'scc_pd']
Total: 691
Train: 483
Val:   103
Test:  105


In [6]:
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(device)
            features = model.encode_image(images, proj_contrast=False, normalize=True)
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy() if hasattr(labels, 'numpy') else np.array(labels))
    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting: 100%|██████████| 8/8 [03:17<00:00, 24.68s/it]


Train: (483, 512)
Extracting val features...


Extracting: 100%|██████████| 2/2 [00:44<00:00, 22.42s/it]


Val: (103, 512)
Extracting test features...


Extracting: 100%|██████████| 2/2 [00:39<00:00, 19.60s/it]

Test: (105, 512)


In [7]:
np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy',   train_labels)
np.save(f'{EMBED_DIR}/val_features.npy',   val_features)
np.save(f'{EMBED_DIR}/val_labels.npy',     val_labels)
np.save(f'{EMBED_DIR}/test_features.npy',  test_features)
np.save(f'{EMBED_DIR}/test_labels.npy',    test_labels)
print(f"Embeddings saved to: {EMBED_DIR}")

Embeddings saved to: /content/drive/MyDrive/PRISM/embeddings/conch/lunghist700


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def compute_ece_multiclass(y_true, y_prob, n_bins=15):
    confidence  = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    correct     = (predictions == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(correct[mask].mean() - confidence[mask].mean())
    return ece / len(y_true)

def stratified_sample(train_l, n, seed):
    """Her siniftan en az 1 ornek garantiler."""
    np.random.seed(seed)
    classes = np.unique(train_l)
    idx = []
    for c in classes:
        c_idx = np.where(train_l == c)[0]
        idx.append(np.random.choice(c_idx, 1)[0])
    already = set(idx)
    remaining = n - len(idx)
    if remaining > 0:
        pool = [i for i in range(len(train_l)) if i not in already]
        extra = np.random.choice(pool, min(remaining, len(pool)), replace=False)
        idx.extend(extra.tolist())
    return np.array(idx[:n])

def run_label_fraction(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    n_classes = len(np.unique(train_l))
    n = max(int(len(train_l) * fraction), n_classes)
    idx = stratified_sample(train_l, n, seed)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    y_prob = clf.predict_proba(test_f)
    y_pred = clf.predict(test_f)
    if y_prob.shape[1] < n_classes:
        full_prob = np.zeros((len(test_f), n_classes))
        for i, c in enumerate(clf.classes_):
            full_prob[:, c] = y_prob[:, i]
        y_prob = full_prob
    return {
        'model': 'CONCH', 'dataset': 'LungHist700',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_l, y_prob, multi_class='ovr', average='macro', labels=list(range(n_classes))),
        'f1':    f1_score(test_l, y_pred, average='macro', zero_division=0),
        'ece':   compute_ece_multiclass(test_l, y_prob),
        'brier': np.mean([brier_score_loss((test_l==c).astype(int), y_prob[:,c]) for c in range(n_classes)]),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.8635 | ECE: 0.3945 | Brier: 0.1148
  1% | seed 123 | AUROC: 0.8351 | ECE: 0.2239 | Brier: 0.1159
  1% | seed 456 | AUROC: 0.8905 | ECE: 0.3666 | Brier: 0.1149
  5% | seed 42 | AUROC: 0.8875 | ECE: 0.1375 | Brier: 0.1087
  5% | seed 123 | AUROC: 0.9236 | ECE: 0.2039 | Brier: 0.1010
  5% | seed 456 | AUROC: 0.9095 | ECE: 0.1803 | Brier: 0.1055
  10% | seed 42 | AUROC: 0.9415 | ECE: 0.2705 | Brier: 0.0949
  10% | seed 123 | AUROC: 0.9371 | ECE: 0.1694 | Brier: 0.0923
  10% | seed 456 | AUROC: 0.9414 | ECE: 0.1285 | Brier: 0.0920
  25% | seed 42 | AUROC: 0.9589 | ECE: 0.2509 | Brier: 0.0768
  25% | seed 123 | AUROC: 0.9562 | ECE: 0.2205 | Brier: 0.0758
  25% | seed 456 | AUROC: 0.9591 | ECE: 0.2409 | Brier: 0.0762
  50% | seed 42 | AUROC: 0.9624 | ECE: 0.2362 | Brier: 0.0639
  50% | seed 123 | AUROC: 0.9637 | ECE: 0.2876 | Brier: 0.0613
  50% | seed 456 | AUROC: 0.9629 | ECE: 0.2031 | Brier: 0.0640
  100% | seed 42 | AUROC: 0.9714 | ECE: 0.2084 | Brier: 0.0518
  1

In [9]:
from scipy.optimize import minimize_scalar

def find_temperature_multiclass(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    n_classes = len(np.unique(train_l))
    n = max(int(len(train_l) * fraction), n_classes)
    idx = stratified_sample(train_l, n, seed)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    val_logits = clf.decision_function(val_f)
    if val_logits.ndim == 1:
        val_logits = np.vstack([-val_logits, val_logits]).T
    T = find_temperature_multiclass(val_logits, val_l)
    test_prob_raw = clf.predict_proba(test_f)
    if test_prob_raw.shape[1] < n_classes:
        full_prob = np.zeros((len(test_f), n_classes))
        for i, c in enumerate(clf.classes_):
            full_prob[:, c] = test_prob_raw[:, i]
        test_prob_raw = full_prob
    ece_raw     = compute_ece_multiclass(test_l, test_prob_raw)
    test_logits = clf.decision_function(test_f)
    if test_logits.ndim == 1:
        test_logits = np.vstack([-test_logits, test_logits]).T
    scaled      = test_logits / T
    exp_s       = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s = exp_s / exp_s.sum(axis=1, keepdims=True)
    if test_prob_s.shape[1] < n_classes:
        full_prob_s = np.zeros((len(test_f), n_classes))
        for i, c in enumerate(clf.classes_):
            full_prob_s[:, c] = test_prob_s[:, i]
        test_prob_s = full_prob_s
    ece_scaled  = compute_ece_multiclass(test_l, test_prob_s)
    return {
        'model': 'CONCH', 'dataset': 'LungHist700',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc':       roc_auc_score(test_l, test_prob_raw, multi_class='ovr', average='macro', labels=list(range(n_classes))),
        'brier_raw':   np.mean([brier_score_loss((test_l==c).astype(int), test_prob_raw[:,c]) for c in range(n_classes)]),
        'brier_scaled':np.mean([brier_score_loss((test_l==c).astype(int), test_prob_s[:,c])   for c in range(n_classes)]),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           0.1000   0.3283      0.1177           0.2106
0.05           0.6057   0.1739      0.1507           0.0233
0.10           0.4111   0.1895      0.1440           0.0455
0.25           0.3151   0.2374      0.1421           0.0953
0.50           0.3957   0.2423      0.0909           0.1514
1.00           0.4785   0.2084      0.1038           0.1046


In [10]:
df.to_csv(f'{RESULTS_DIR}/conch_lunghist700_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/conch_lunghist700_temperature_scaling.csv', index=False)
print("Results saved!")
print(f"  {RESULTS_DIR}/conch_lunghist700_results.csv")
print(f"  {RESULTS_DIR}/conch_lunghist700_temperature_scaling.csv")

Results saved!
  /content/drive/MyDrive/PRISM/results/conch_lunghist700_results.csv
  /content/drive/MyDrive/PRISM/results/conch_lunghist700_temperature_scaling.csv
